# ============================================================
# [Data Integration] Dental Offices in Berlin
# ============================================================
#### **Install dependencies (if not already installed):**
 - Datenverarbeitung
`conda install -c conda-forge pandas=2.3.3 numpy=1.26.4 -y`
 - **Geospatial**
`conda install -c conda-forge geopy osmnx geopandas=0.14 shapely=2.1.2 fiona=1.9.4 pyproj=3.6.2 -y`
 - **SQL / Datenbank**
`conda install -c conda-forge sqlalchemy=2.0.45 psycopg2=2.9.9 -y`
 - **Utilities**
`conda install -c conda-forge tqdm=4.66.1 python-dotenv=1.0.0 -y`

# ============================================================
# 0. Imports
# ============================================================

In [132]:
# =============================================================
# Environment & Standard Library
# =============================================================
import os
import re
import pickle
import warnings
from pathlib import Path
from time import sleep

# =============================================================
# Data Processing & Scientific Computing
# =============================================================
import numpy as np
import pandas as pd

# =============================================================
# Geospatial Processing
# =============================================================
import geopandas as gpd
import osmnx as ox

from shapely.geometry import Point
from shapely.ops import unary_union

# =============================================================
# Geocoding
# =============================================================
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# =============================================================
# Database & ORM
# =============================================================
import psycopg2
from psycopg2 import sql as psql

from sqlalchemy import create_engine, text
from sqlalchemy.exc import IntegrityError

from dotenv import load_dotenv

# =============================================================
# Utilities & Helpers
# =============================================================
from difflib import SequenceMatcher
from tqdm import tqdm

# =============================================================
# Configuration
# =============================================================
warnings.filterwarnings("ignore")

WGS84 = "EPSG:4326"

# ============================================================
# Step I: Research & Data Modelling
# ============================================================
## 1.1. Data Extraction & Initial Inspection
## ============================================================

In [133]:
# 1.1 OSM Settings
ox.settings.use_cache = True
ox.settings.log_console = True

# 1.2 Fetch Dental Offices from OpenStreetMap
tags = {"amenity": "dentist"}

dental_offices_osm = ox.features.features_from_place(
    "Berlin, Germany",
    tags=tags
)

print(f"Number of dental office entries fetched: {len(dental_offices_osm)}")
print(dental_offices_osm.head(3).T.to_string())
print(dental_offices_osm['healthcare:speciality'].value_counts())

Number of dental office entries fetched: 798
element                                                  node                                                                                                                                                   
id                                                  304183504                                                                       313539258                                                          325161442
geometry                         POINT (13.612096 52.5114112)                                                   POINT (13.3553052 52.5488382)                                      POINT (13.1804772 52.5088434)
addr:city                                              Berlin                                                                          Berlin                                                                NaN
addr:country                                               DE                                                          

In [134]:
# Save raw data for reproducibility
dental_offices_osm.to_csv("../sources/raw_osm_dental_offices_v_01_19_2026.csv", index=False)
dental_offices_osm.to_file("../sources/raw_osm_dental_offices_v_01_19_2026.geojson", driver="GeoJSON")

# Inspect dataset
dental_offices_osm.info()
print(dental_offices_osm.columns.tolist())

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Columns: 104 entries, geometry to type
dtypes: geometry(1), object(103)
memory usage: 689.8+ KB
['geometry', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'health_facility:type', 'health_specialty:dentistry', 'healthcare', 'medical_system:western', 'office', 'operator', 'description', 'level', 'name', 'opening_hours', 'wheelchair', 'phone', 'toilets:wheelchair', 'website', 'contact:website', 'contact:email', 'contact:fax', 'contact:phone', 'check_date:opening_hours', 'check_date', 'email', 'fax', 'healthcare:speciality', 'air_conditioning', 'internet_access', 'internet_access:fee', 'payment:bank_transfer', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'payment:paypal', 'source', 'toilets', 'entrance', 'opening_hours:signed', 'wheelchair:description', 'emergency', 'name:de', 'name:en', 'payment:cont

## ============================================================
## 1.2. Data Selection & Column Standardization
## ============================================================

In [135]:
# Select key columns relevant for dental offices
columns = [
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "level",
    "addr:floor",
    "description",
    "opening_hours",
    "check_date:opening_hours",
    "check_date",
    "healthcare:speciality",
    "wheelchair",
    "wheelchair:description",
    "toilets:wheelchair",
    "phone",
    "contact:website", 
    "contact:email", 
    "contact:phone",
    "email",
    "url",
    "website",
    "geometry",
    "health_facility:type",
    "health_specialty:oral_surgery",
    "health_specialty:orthodontics",
    "health_specialty:periodontology"
]

# Filter the dataset to keep only the selected columns
dental_offices = dental_offices_osm[[c for c in columns if c in dental_offices_osm.columns]].copy()



## ============================================================
## 1.3. Speciality Mapping
## ============================================================

In [136]:
# Mapping of OSM health specialty columns to standardized speciality names
# Only consider values marked as "yes" or "main"
special_cols = {
    "health_specialty:oral_surgery": "oral_surgery",
    "health_specialty:orthodontics": "orthodontics",
    "health_specialty:periodontology": "periodontology"
}

def compute_speciality(row):
    """
    Determine standardized speciality for a dental office.
    
    Priority:
    1. Use 'healthcare:speciality' if non-empty
    2. Check boolean-style specialty columns ("yes" or "main")
    3. Default to "None"
    """
    val = row.get("healthcare:speciality")
    if pd.notna(val) and str(val).strip() != "":
        return str(val).strip()
    
    for col, name in special_cols.items():
        cell = row.get(col)
        if pd.notna(cell) and str(cell).lower() in ["yes", "main"]:
            return name
    
    return None

# Apply speciality mapping
dental_offices["speciality"] = dental_offices.apply(compute_speciality, axis=1)

# Drop original speciality columns to avoid redundancy
dental_offices.drop(
    columns=["healthcare:speciality"] + list(special_cols.keys()), inplace=True
)

dental_offices.head()

name     addr:street addr:housenumber  \
element id                                                                
node    304183504                  NaN  Hönower Straße               75   
        313539258  Zahnzentrum Wedding    Müllerstraße              34a   
        325161442             A. Nejad             NaN              NaN   
        345236220    Dr. Beate Lengert  Kurfürstendamm              218   
        391394177      Serpil Hartfiel  Kollwitzstraße               77   

                  addr:postcode addr:city level addr:floor description  \
element id                                                               
node    304183504         12623    Berlin   NaN        NaN         NaN   
        313539258         13353    Berlin     2        NaN    Zahnarzt   
        325161442           NaN       NaN   NaN        NaN         NaN   
        345236220         10719    Berlin   NaN        NaN         NaN   
        391394177         10435    Berlin   NaN        NaN         NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   
        345236220                                                NaN   
        391394177  Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...   

                  check_date:opening_hours  ...             phone  \
element id                                  ...                     
node    304183504                      NaN  ...               NaN   
        313539258                      NaN  ...               NaN   
        325161442                      NaN  ...  +49 30 361 91 06   
        345236220                      NaN  ...               NaN   
        391394177                      NaN  ...               NaN   

                                           contact:website contact:email  \
element id                                                                 
node    304183504                                      NaN           NaN   
        313539258                                      NaN           NaN   
        325161442                                      NaN           NaN   
        345236220                                      NaN           NaN   
        391394177  https://www.zahnarztpraxis-hartfiel.de/           NaN   

                  contact:phone email  url                          website  \
element id                                                                    
node    304183504           NaN   NaN  NaN                              NaN   
        313539258           NaN   NaN  NaN                              NaN   
        325161442           NaN   NaN  NaN                              NaN   
        345236220           NaN   NaN  NaN  http://www.dr-beate-lengert.de/   
        391394177           NaN   NaN  NaN                              NaN   

                                    geometry health_facility:type speciality  
element id                                                                    
node    304183504   POINT (13.6121 52.51141)               office       None  
        313539258  POINT (13.35531 52.54884)                  NaN       None  
        325161442  POINT (13.18048 52.50884)                  NaN       None  
        345236220  POINT (13.32814 52.50272)                  NaN       None  
        391394177  POINT (13.41899 52.53755)                  NaN       None  

[5 rows x 24 columns]

## ============================================================
## 1.4. Geospatial Normalization & Attribute Consolidation for Dental Offices
## ============================================================

In [137]:
# ============================================================
# Geometry Normalization
# ============================================================
# Ensure that all geometries are Points.
# For non-point geometries, use a representative point.
dental_offices["geometry"] = dental_offices["geometry"].apply(
    lambda g: g if g.geom_type == "Point" else g.representative_point()
)
# Extract latitude and longitude
dental_offices["latitude"] = dental_offices.geometry.y
dental_offices["longitude"] = dental_offices.geometry.x
# ============================================================
# Address Column Standardization
# ============================================================
# Rename OpenStreetMap-style address tags to standardized column names
dental_offices = dental_offices.rename(
    columns={
        "addr:street": "street",
        "addr:housenumber": "housenumber",
        "addr:postcode": "postcode",
        "addr:city": "city",
    }
)

# Inspect dataframe structure
dental_offices.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   name                      767 non-null    object  
 1   street                    582 non-null    object  
 2   housenumber               582 non-null    object  
 3   postcode                  535 non-null    object  
 4   city                      527 non-null    object  
 5   level                     90 non-null     object  
 6   addr:floor                2 non-null      object  
 7   description               24 non-null     object  
 8   opening_hours             602 non-null    object  
 9   check_date:opening_hours  177 non-null    object  
 10  check_date                140 non-null    object  
 11  wheelchair                286 non-null    object  
 12  wheelchair:description    8 non-null      object  
 13  toilets:w

In [138]:
# ============================================================
# Exploratory Value Inspection (Data Profiling)
# ============================================================
# Useful for understanding data variability before consolidation
INSPECTION_COLUMNS = [
    "addr:floor",
    "description",
    "opening_hours",
    "check_date",
    "check_date:opening_hours",
    "wheelchair",
    "wheelchair:description",
    "toilets:wheelchair",
    "phone",
    "contact:phone",
    "website",
    "contact:website",
    "email",
    "contact:email",
    "url",
    "health_facility:type",
]

for col in INSPECTION_COLUMNS:
    if col in dental_offices.columns:
        print(f"{col}: {dental_offices[col].unique()}")

addr:floor: [nan '1']
description: [nan 'Zahnarzt' 'Zahnärzte & Kieferorthopädie, Aufgang G'
 'Zahnarztpraxis Ursula Frömming, Ruth Bodenheimer'
 'Praxis für ästhetisch-rekonstruktive Zahnmedizin und Implantologie'
 'Kieferorthopädin'
 'Öffnungszeiten: Mo - Fr 9:00 - 19:00 Uhr Fr nur mit Termin'
 'Zahnarztpraxis Reinickendorf, Zahnarzt Reinickendorf, Implantation Reinickendorf, Zahnarzt Wittenau'
 'Dipl.-Stomatologin' 'Dr. med. dent. Peter Zemlin'
 'Dr. med. dent. Nihat Akdeniz, Zahnärztin Sümeyra Pesen und Zahnarzt Tarkan Cangöz'
 'Gemeinschaftspraxis Andreas und Susanna Fleck'
 'Mit fünf spezialisierten Zahnärztinnen und Zahnärzten, sowie einem Mund-, Kiefer- und Gesichtschirurg, einem zahntechnischen Meisterlabor im Haus und vier Prophylaxeassistentinnen bieten wir Ihnen hier das gesamte zahnärztliche Leistungsspektrum an.'
 'Koch & Gruner-Koch'
 'Kieferorthopädie in Berlin. Zum Leistungsspektrum gehören unter anderem: lose Zahnspangen, festen Zahnspange, Retentionsgeräte, durchsich

In [139]:
# ============================================================
# Column Merge Utilities
# ============================================================
def merge_columns(val1, val2, sep=" | "):
    """
    Merge two column values using defined rules:
    - If both values are missing → None
    - If only one value exists → return it
    - If both values are equal → return one
    - Otherwise → concatenate using separator
    """
    if pd.isna(val1) and pd.isna(val2):
        return None
    if pd.isna(val1):
        return val2
    if pd.isna(val2):
        return val1
    if str(val1) == str(val2):
        return val1
    return f"{val1}{sep}{val2}"


def merge_check_dates(opening_val, general_val, sep=" | "):
    """
    Specialized merge logic for check_date:
    - Preserve semantic context (opening hours vs general)
    """
    values = []

    if not pd.isna(opening_val):
        values.append(f"opening hours: {opening_val}")

    if not pd.isna(general_val):
        values.append(f"general: {general_val}")

    return sep.join(values) if values else None


def merge_accessibility(wheelchair_val, wheelchair_toilets_val, sep=" | "):
    """
    Consolidate accessibility-related attributes into a single field.
    """
    values = []

    if not pd.isna(wheelchair_val):
        values.append(f"wheelchair: {wheelchair_val}")

    if not pd.isna(wheelchair_toilets_val):
        values.append(f"wheelchair toilets: {wheelchair_toilets_val}")

    return sep.join(values) if values else None


# ============================================================
# Attribute Consolidation
# ============================================================
dental_offices["level"] = dental_offices.apply(
    lambda x: merge_columns(x.get("level"), x.get("addr:floor")),
    axis=1,
)

dental_offices["accessibility"] = dental_offices.apply(
    lambda x: merge_accessibility(x.get("wheelchair"), x.get("toilets:wheelchair")),
    axis=1,
)

dental_offices["description"] = dental_offices.apply(
    lambda x: merge_columns(x.get("description"), x.get("wheelchair:description")),
    axis=1,
)

dental_offices["phone"] = dental_offices.apply(
    lambda x: merge_columns(x.get("phone"), x.get("contact:phone")),
    axis=1,
)

dental_offices["email"] = dental_offices.apply(
    lambda x: merge_columns(x.get("email"), x.get("contact:email")),
    axis=1,
)

dental_offices["website"] = dental_offices.apply(
    lambda x: merge_columns(
        merge_columns(x.get("website"), x.get("contact:website")),
        x.get("url"),
    ),
    axis=1,
)

dental_offices["check_date"] = dental_offices.apply(
    lambda x: merge_check_dates(
        x.get("check_date:opening_hours"),
        x.get("check_date"),
    ),
    axis=1,
)

# ============================================================
# Final Schema Selection
# ============================================================
FINAL_COLUMNS = [
    "name",
    "street",
    "housenumber",
    "postcode",
    "city",
    "level",
    "description",
    "opening_hours",
    "check_date",
    "accessibility",
    "phone",
    "email",
    "website",
    "geometry",
    "health_facility:type",
    "speciality",
    "latitude",
    "longitude",
]

dental_offices = dental_offices[FINAL_COLUMNS]

# Rename remaining OSM-style column
dental_offices = dental_offices.rename(
    columns={"health_facility:type": "office_type"}
)

# Final validation
print(dental_offices.info())
dental_offices.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   name           767 non-null    object  
 1   street         582 non-null    object  
 2   housenumber    582 non-null    object  
 3   postcode       535 non-null    object  
 4   city           527 non-null    object  
 5   level          91 non-null     object  
 6   description    32 non-null     object  
 7   opening_hours  602 non-null    object  
 8   check_date     284 non-null    object  
 9   accessibility  286 non-null    object  
 10  phone          460 non-null    object  
 11  email          137 non-null    object  
 12  website        426 non-null    object  
 13  geometry       798 non-null    geometry
 14  office_type    23 non-null     object  
 15  speciality     112 non-null    object  
 16  latitude       798 non-null    float64 

name          street housenumber postcode  \
element id                                                                    
node    304183504                  NaN  Hönower Straße          75    12623   
        313539258  Zahnzentrum Wedding    Müllerstraße         34a    13353   
        325161442             A. Nejad             NaN         NaN      NaN   
        345236220    Dr. Beate Lengert  Kurfürstendamm         218    10719   
        391394177      Serpil Hartfiel  Kollwitzstraße          77    10435   

                     city level description  \
element id                                    
node    304183504  Berlin  None        None   
        313539258  Berlin     2    Zahnarzt   
        325161442     NaN  None        None   
        345236220  Berlin  None        None   
        391394177  Berlin  None        None   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   
        345236220                                                NaN   
        391394177  Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...   

                  check_date                             accessibility  \
element id                                                               
node    304183504       None                                      None   
        313539258       None                           wheelchair: yes   
        325161442       None  wheelchair: yes | wheelchair toilets: no   
        345236220       None                                      None   
        391394177       None                            wheelchair: no   

                              phone email  \
element id                                  
node    304183504              None  None   
        313539258              None  None   
        325161442  +49 30 361 91 06  None   
        345236220              None  None   
        391394177              None  None   

                                                   website  \
element id                                                   
node    304183504                                     None   
        313539258                                     None   
        325161442                                     None   
        345236220          http://www.dr-beate-lengert.de/   
        391394177  https://www.zahnarztpraxis-hartfiel.de/   

                                    geometry office_type speciality  \
element id                                                            
node    304183504   POINT (13.6121 52.51141)      office       None   
        313539258  POINT (13.35531 52.54884)         NaN       None   
        325161442  POINT (13.18048 52.50884)         NaN       None   
        345236220  POINT (13.32814 52.50272)         NaN       None   
        391394177  POINT (13.41899 52.53755)         NaN       None   

                    latitude  longitude  
element id                               
node    304183504  52.511411  13.612096  
        313539258  52.548838  13.355305  
        325161442  52.508843  13.180477  
        345236220  52.502722  13.328137  
        391394177  52.537547  13.418994

## ============================================================
## 1.5. Neighborhood Assignment via Spatial Join
## ============================================================

In [140]:

# Load neighborhoods GeoJSON (Berlin LOR Ortsteile)
lor_path = Path('../../mapping/lor_ortsteile.geojson')
neighborhoods = gpd.read_file(lor_path).to_crs(WGS84)

# Rename for consistency
neighborhoods = neighborhoods.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
})

# Create GeoDataFrame for dental offices
dental_gdf = gpd.GeoDataFrame(dental_offices, geometry='geometry', crs=WGS84)
print(f"✓ Created GeoDataFrame with {len(dental_gdf)} dental offices")


✓ Created GeoDataFrame with 798 dental offices


In [141]:
# Spatial join to assign neighborhoods
df_with_districts = gpd.sjoin(
    dental_gdf,
    neighborhoods[["district", "neighborhood", "neighborhood_id", "geometry"]],
    how="left",
    predicate="within"
)

# Drop unnecessary columns from spatial join
df_final = df_with_districts.drop(columns=["index_right"])
df_final.head(3)

name          street housenumber postcode  \
element id                                                                    
node    304183504                  NaN  Hönower Straße          75    12623   
        313539258  Zahnzentrum Wedding    Müllerstraße         34a    13353   
        325161442             A. Nejad             NaN         NaN      NaN   

                     city level description  \
element id                                    
node    304183504  Berlin  None        None   
        313539258  Berlin     2    Zahnarzt   
        325161442     NaN  None        None   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   

                  check_date                             accessibility  ...  \
element id                                                              ...   
node    304183504       None                                      None  ...   
        313539258       None                           wheelchair: yes  ...   
        325161442       None  wheelchair: yes | wheelchair toilets: no  ...   

                  email website                   geometry office_type  \
element id                                                               
node    304183504  None    None   POINT (13.6121 52.51141)      office   
        313539258  None    None  POINT (13.35531 52.54884)         NaN   
        325161442  None    None  POINT (13.18048 52.50884)         NaN   

                  speciality   latitude  longitude             district  \
element id                                                                
node    304183504       None  52.511411  13.612096  Marzahn-Hellersdorf   
        313539258       None  52.548838  13.355305                Mitte   
        325161442       None  52.508843  13.180477              Spandau   

                   neighborhood neighborhood_id  
element id                                       
node    304183504     Mahlsdorf            1004  
        313539258       Wedding            0105  
        325161442  Wilhelmstadt            0509  

[3 rows x 21 columns]

## ============================================================
## 1.6. District ID Mapping
## ============================================================

In [142]:
# Official Berlin district identifiers
DISTRICT_MAPPING = {
    "Mitte": "11001001",
    "Friedrichshain-Kreuzberg": "11002002",
    "Pankow": "11003003",
    "Charlottenburg-Wilmersdorf": "11004004",
    "Spandau": "11005005",
    "Steglitz-Zehlendorf": "11006006",
    "Tempelhof-Schöneberg": "11007007",
    "Neukölln": "11008008",
    "Treptow-Köpenick": "11009009",
    "Marzahn-Hellersdorf": "11010010",
    "Lichtenberg": "11011011",
    "Reinickendorf": "11012012",
}

# Map district names to standardized district IDs
df_final["district_id"] = df_final["district"].map(DISTRICT_MAPPING)

# ============================================================
# Data Quality Check: Unmapped Districts
# ============================================================
unmapped_districts = (
    df_final.loc[~df_final["district"].isin(DISTRICT_MAPPING.keys()), "district"]
    .dropna()
    .unique()
)

if len(unmapped_districts) > 0:
    print("⚠️ Unmapped districts detected:", unmapped_districts)

# Ensure district_id is stored as string (SQL-safe)
df_final["district_id"] = df_final["district_id"].astype("string")

## ============================================================
## 1.7 Final Record Indexing & Identifier Standardization
## ============================================================

In [143]:
# Reset the DataFrame index to ensure a clean, continuous index
df_final = df_final.reset_index()

# Remove technical or intermediate columns that are no longer needed
# and rename the primary identifier to a domain-specific name
df_final = df_final.drop(columns=["element"]).rename(
    columns={"id": "id"}
)

# Ensure the dental office identifier is stored as a string
# (recommended for database storage and interoperability)
df_final["id"] = df_final["id"].astype("string")

# ============================================================
# Basic Record Integrity Checks
# ============================================================

print(df_final.shape[0], "total dental office records after processing.")
print(
    df_final["id"].nunique(),
    "unique dental offices IDs assigned."
)


798 total dental office records after processing.
798 unique dental offices IDs assigned.


## ============================================================
## 1.8. Floor / Level Standardization
## ============================================================

In [144]:
# Locale-specific floor mappings

LEVEL_MAP_DEU = {
    "-1": "UG",
    "0": "EG",
    "1": "1.OG",
    "2": "2.OG",
    "3": "3.OG",
    "4": "4.OG",
    "5": "5.OG",
    "6": "6.OG",
    "7": "7.OG",
    "8": "8.OG",
    "9": "9.OG",
    "10": "10.OG",
}

LEVEL_MAP_UK = {
    "-1": "Basement",
    "0": "Ground Floor",
    "1": "First Floor",
    "2": "Second Floor",
    "3": "Third Floor",
    "4": "Fourth Floor",
    "5": "Fifth Floor",
    "6": "Sixth Floor",
}

LEVEL_MAP_US = {
    "-1": "Basement",
    "0": "First Floor",
    "1": "Second Floor",
    "2": "Third Floor",
    "3": "Fourth Floor",
    "4": "Fifth Floor",
    "5": "Sixth Floor",
    "6": "Seventh Floor",
}

# Select active level mapping (DEU / UK / US)
LEVEL_MAP = LEVEL_MAP_DEU


def format_level(level):
    """
    Convert numeric or coded floor levels into standardized,
    human-readable labels based on the selected locale.
    """
    if pd.isna(level):
        return None

    level_str = str(level).strip()
    return LEVEL_MAP.get(level_str, level_str)


## ============================================================
## 1.9. Fallback Address Enrichment via Reverse Geocoding (Nominatim)
## ============================================================

In [145]:
# -------------------------------------------------------------
# Initialize the Nominatim geocoder with a custom user agent
# Nominatim requires a user_agent string to identify the application.
# Using a descriptive name helps comply with their usage policy.
# -------------------------------------------------------------

geolocator = Nominatim(user_agent="berlin_dental_offices_project", timeout=10)
# RateLimiter: 1 request per second to respect public Nominatim policy
geocode_rate_limited = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# -----------------------------
# Initialize local cache for reverse geocoding results
# -----------------------------
cache_file = "nominatim_reverse_cache.pkl"

try:
    with open(cache_file, "rb") as f:
        reverse_cache = pickle.load(f)
except FileNotFoundError:
    reverse_cache = {}

# -----------------------------
# Debug info before geocoding
# -----------------------------
print(f"[INFO] Missing house numbers before Nominatim: {df_final['housenumber'].isna().sum()}")
print(f"[INFO] Missing streets before Nominatim: {df_final['street'].isna().sum()}")

# -----------------------------
# Function: reverse geocode with shift, retry, and caching
# -----------------------------
def reverse_geocode_address_components(lat, lon, max_attempts=10, shift_deg=0.0001, retry_count=3):
    """
    Reverse-geocode latitude and longitude into structured address components.
    Uses radius search with small shifts and retries on timeout errors.
    Caches results to avoid repeated API calls.

    Parameters
    ----------
    lat, lon : float
        Coordinates to reverse-geocode
    max_attempts : int
        Number of shifted coordinate attempts (~10 meters each)
    shift_deg : float
        Degree shift applied for nearby search
    retry_count : int
        Number of retries per shifted coordinate in case of connection timeout

    Returns
    -------
    dict
        Dictionary with keys: street, housenumber, postcode, city
        Missing values returned as pd.NA
    """

    # Round coordinates for cache key (~0.1 m precision)
    key = (round(lat, 6), round(lon, 6))

    # Return cached result if available
    if key in reverse_cache:
        return reverse_cache[key]

    # Define coordinate shifts: original + small N/S/E/W shifts
    shifts = [(0, 0), (shift_deg, 0), (-shift_deg, 0), (0, shift_deg), (0, -shift_deg)]

    for dx, dy in shifts[:max_attempts]:
        for attempt in range(retry_count):
            try:
                # Perform reverse geocoding (rate-limited)
                location = geocode_rate_limited(
                    (lat + dx, lon + dy),
                    exactly_one=True,
                    language="en"
                )

                # If result found, extract structured address
                if location and location.raw.get("address"):
                    address = location.raw.get("address", {})

                    street_name = address.get("road") or address.get("place") or pd.NA
                    result = {
                        "street": street_name,
                        "housenumber": address.get("house_number", pd.NA),
                        "postcode": address.get("postcode", pd.NA),
                        "city": (
                            address.get("city")
                            or address.get("town")
                            or address.get("village")
                            or address.get("municipality")
                            or address.get("suburb")
                            or pd.NA
                        ),
                    }

                    # Save to cache and return
                    reverse_cache[key] = result
                    return result

            except Exception as e:
                # Warn and retry
                print(f"[WARN] Reverse geocode failed for ({lat+dx}, {lon+dy}), attempt {attempt+1}/{retry_count}: {e}")
                sleep(2)  # small pause before retry

    # If all attempts fail, store and return empty components
    result = {"street": pd.NA, "housenumber": pd.NA, "postcode": pd.NA, "city": pd.NA}
    reverse_cache[key] = result
    return result

# -----------------------------
# Apply reverse geocoding to missing address components
# -----------------------------
address_cols = ["street", "housenumber", "postcode", "city"]

# Mask: only rows with missing components and valid coordinates
mask = (
    df_final[address_cols].isna().any(axis=1) &
    df_final["latitude"].notna() &
    df_final["longitude"].notna()
)

# Iterate over masked rows
for idx, row in tqdm(df_final.loc[mask].iterrows(), total=mask.sum()):
    components = reverse_geocode_address_components(row["latitude"], row["longitude"])
    for col in address_cols:
        # Only fill missing values
        if pd.isna(row[col]) and pd.notna(components[col]):
            df_final.at[idx, col] = components[col]

# -----------------------------
# Save cache to disk for future runs
# -----------------------------
with open(cache_file, "wb") as f:
    pickle.dump(reverse_cache, f)

# -----------------------------
# Debug info after geocoding
# -----------------------------
print(f"[INFO] Missing house numbers after Nominatim: {df_final['housenumber'].isna().sum()}")
print(f"[INFO] Missing streets after Nominatim: {df_final['street'].isna().sum()}")
# -------------------------------------------------------------
# Note on the radius search:
# If no result is found at the original coordinates, the function
# shifts the coordinates slightly (~10 meters) in multiple directions
# to perform a nearby lookup.
#
# To maximize coverage around each point, the parameter `max_attempts`
# was set to 10, allowing the function to probe multiple nearby locations
# in order to increase the likelihood of finding an associated address.
#
# This approach helps capture points that lie between buildings or street
# geometries and would otherwise not be resolved by Nominatim.
#
# Additionally, if no 'road' field is returned by Nominatim, the function
# falls back to using the 'place' field to populate the 'street' value,
# ensuring that addresses stored under alternative address tags are
# also captured.
# -------------------------------------------------------------


[INFO] Missing house numbers before Nominatim: 216
[INFO] Missing streets before Nominatim: 216


100%|██████████| 280/280 [00:00<00:00, 18533.09it/s]

[INFO] Missing house numbers after Nominatim: 187
[INFO] Missing streets after Nominatim: 1


## ============================================================
## 1.10. Address Formatting
## ============================================================

In [146]:
def format_address(row):
    """
    Construct full address string with optional floor and neighborhood.
    If all core address fields are missing, return 'Unknown'.
    """
    # If all core fields are missing
    if all(pd.isna(row[col]) for col in ['street', 'housenumber', 'level', 'postcode', 'city']):
        return pd.NA
    
    # Prepare components
    street = row['street'] if pd.notna(row['street']) else ""
    housenumber = row['housenumber'] if pd.notna(row['housenumber']) else ""
    postcode = row['postcode'] if pd.notna(row['postcode']) else ""
    city = row['city'] if pd.notna(row['city']) else ""
    neighborhood = row['neighborhood'] if pd.notna(row['neighborhood']) else ""
    
    # Floor formatting
    level_str = format_level(row['level'])
    
    # Construct address parts
    parts = [f"{street} {housenumber}".strip()]
    if level_str:
        parts.append(level_str)
    city_line = f"{postcode} {city}".strip()
    if neighborhood:
        city_line += f"-{neighborhood}"
    parts.append(city_line)
    
    return ", ".join([p for p in parts if p])

# Apply formatted address
df_final["address"] = df_final.apply(format_address, axis=1)
# Quick check of results
df_final["address"].isna().sum()
df_final[["latitude", "longitude", "address"]].head(10)



,latitude,longitude,address
0,52.511411,13.612096,"Hönower Straße 75, 12623 Berlin-Mahlsdorf"
1,52.548838,13.355305,"Müllerstraße 34a, 2.OG, 13353 Berlin-Wedding"
2,52.508843,13.180477,"Gatower Straße, 13595 Berlin-Wilhelmstadt"
3,52.502722,13.328137,"Kurfürstendamm 218, 10719 Berlin-Charlottenburg"
4,52.537547,13.418994,"Kollwitzstraße 77, 10435 Berlin-Prenzlauer Berg"
5,52.384968,13.404870,"Goltzstraße, 12307 Berlin-Lichtenrade"
6,52.466306,13.385948,"Tempelhofer Damm 143, 12099 Berlin-Tempelhof"
7,52.451063,13.385178,"Mariendorfer Damm, 12109 Berlin-Mariendorf"
8,52.525158,13.310129,"Ilsenburger Straße, 10589 Berlin-Charlottenburg"
9,52.541379,13.353790,"Sprengelstraße, 13353 Berlin-Wedding"


In [147]:
df_final[df_final['street'].isna()] 

,id,name,street,housenumber,postcode,city,level,description,opening_hours,check_date,...,geometry,office_type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id,address
131,3353229503,Dentalzentrum Pankow,NaN,1,13187,Berlin,None,None,"Mo-We 10:30-18:00; Th,Fr 08:30-16:00",opening hours: 2025-11-02,...,POINT (13.41094 52.56735),NaN,None,52.567345,13.41094,Pankow,Pankow,0307,11003003,"1, 13187 Berlin-Pankow"


In [148]:
# Hardcoding the street names for entries with missing street information
hardcoded_streets = "Garbátyplatz"
df_final.loc[df_final['name'] == "Dentalzentrum Pankow", 'street'] = hardcoded_streets
df_final[df_final['street'].isna()] 

,id,name,street,housenumber,postcode,city,level,description,opening_hours,check_date,...,geometry,office_type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id,address


## ============================================================
## 1.11. First Data Overview / Quality Check (Columns, Types, Missing Values)
## ============================================================

In [149]:
# Missing Values
missing = df_final.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_final) * 100).round(1)
print('Data types:')
display(df_final.dtypes)
print(f'Dental Offices Data: Rows: {df_final.shape[0]}, Columns: {df_final.shape[1]}')
print("Missing Values Summary:")
pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

Data types:


id                 string[python]
name                       object
street                     object
housenumber                object
postcode                   object
city                       object
level                      object
description                object
opening_hours              object
check_date                 object
accessibility              object
phone                      object
email                      object
website                    object
geometry                 geometry
office_type                object
speciality                 object
latitude                  float64
longitude                 float64
district                   object
neighborhood               object
neighborhood_id            object
district_id        string[python]
address                    object
dtype: object

Dental Offices Data: Rows: 798, Columns: 24
Missing Values Summary:


,missing_count,missing_pct
office_type,775,97.1
description,766,96.0
level,707,88.6
speciality,686,86.0
email,661,82.8
check_date,514,64.4
accessibility,512,64.2
website,372,46.6
phone,338,42.4
opening_hours,196,24.6


# ============================================================
# Step II: Data Transformation & Preprocessing
# ============================================================
## 2.1. Dental Office Deduplication Pipeline
## =============================================================
**This pipeline deduplicates dental office records based on:**
- 1. Normalized name + address
- 2. Rounded coordinates (geom_key) ~1m precision
- 3. Final centroid proximity (~10 m) + name check
It produces a duplicate report for diagnostics.

In [150]:
df_final_dupl = (
    df_final
    .groupby(["name", "address"], dropna=False)
    .size()
    .reset_index(name="count_duplicates")
    .sort_values("count_duplicates", ascending=False)
)

df_final_dupl.head()


,name,address,count_duplicates
789,NaN,"Rheinsteinstraße 1, 10318 Berlin-Karlshorst",2
488,Wunschlächeln,"Gotthardstraße 27, 13407 Berlin-Reinickendorf",2
0,99 Zahnspangen,"Albrechtrstraße 99, 12167 Berlin-Steglitz",1
534,Zahnarztpraxis Bismarckstr. 49,"Bismarckstraße 49, 12169 Berlin-Steglitz",1
524,Zahnarztpraxis,"Eichborndamm 18, 13403 Berlin-Reinickendorf",1


In [151]:
df_final[df_final['name']=="Wunschlächeln"]

,id,name,street,housenumber,postcode,city,level,description,opening_hours,check_date,...,geometry,office_type,speciality,latitude,longitude,district,neighborhood,neighborhood_id,district_id,address
49,1379836612,Wunschlächeln,Gotthardstraße,27,13407,Berlin,None,None,Mo-Th 08:00-20:00; Fr 08:00-14:00,None,...,POINT (13.34549 52.56626),NaN,None,52.566255,13.345490,Reinickendorf,Reinickendorf,1201,11012012,"Gotthardstraße 27, 13407 Berlin-Reinickendorf"
276,5890350141,Wunschlächeln,Gotthardstraße,27,13407,Berlin,None,None,NaN,None,...,POINT (13.34615 52.56659),NaN,None,52.566592,13.346145,Reinickendorf,Reinickendorf,1201,11012012,"Gotthardstraße 27, 13407 Berlin-Reinickendorf"


In [152]:
# -------------------------------------------------------------
# Step 0: Prepare working copy
# -------------------------------------------------------------
df_dupl = df_final.copy()

# -------------------------------------------------------------
# Step 1: Normalize Name & Address
# -------------------------------------------------------------
def normalize_text(val, field_type="name"):
    """
    Normalize string values for deduplication.
    """
    if pd.isna(val):
        return "Unknown" if field_type == "name" else None

    val = str(val).lower().strip()
    if field_type == "address":
        val = val.replace("straße", "str").replace("str.", "str").replace("str ", "str")
        val = val.replace(".", "").replace(",", "")
        val = " ".join(val.split())
    elif field_type == "name":
        val = val.replace(".", "").replace(",", "").replace("-", " ").replace("&", "and")
        val = " ".join(val.split())
    return val

df_dupl["name"] = df_dupl["name"].apply(lambda x: normalize_text(x, "name"))
df_dupl["address"] = df_dupl["address"].apply(lambda x: normalize_text(x, "address"))


# -------------------------------------------------------------
# Step 2: Helper function to merge duplicate rows safely
# -------------------------------------------------------------
def merge_rows_safely(group: pd.DataFrame) -> pd.Series:
    merged = group.iloc[0].copy()
    for col in group.columns:
        values = group[col].dropna()
        if not values.empty:
            merged[col] = values.iloc[0]
    if {"latitude", "longitude"}.issubset(group.columns):
        merged["latitude"] = group["latitude"].dropna().mean()
        merged["longitude"] = group["longitude"].dropna().mean()
    return merged

# -------------------------------------------------------------
# Step 3: Deduplicate by normalized Name + Address
# -------------------------------------------------------------
df_final_deduplicated = (
    df_dupl
    .groupby(["name", "address"], as_index=False)
    .apply(merge_rows_safely)
    .reset_index(drop=True)
)

# -------------------------------------------------------------
# Step 4: Convert to GeoDataFrame
# -------------------------------------------------------------
gdf = gpd.GeoDataFrame(
    df_final_deduplicated,
    geometry=gpd.points_from_xy(df_final_deduplicated.longitude, df_final_deduplicated.latitude),
    crs=WGS84
)

# -------------------------------------------------------------
# Step 5: Geospatial deduplication using rounded coordinates
# (~1 m precision)
# -------------------------------------------------------------
gdf["geom_key"] = gdf.geometry.apply(lambda g: f"{round(g.x, 5)}|{round(g.y, 5)}")
df_geo_dedup = (
    gdf.drop(columns="geometry")
       .groupby("geom_key", as_index=False)
       .apply(merge_rows_safely)
       .reset_index(drop=True)
)
gdf = gpd.GeoDataFrame(
    df_geo_dedup,
    geometry=gpd.points_from_xy(df_geo_dedup.longitude, df_geo_dedup.latitude),
    crs=WGS84
)

# -------------------------------------------------------------
# Step 6: Final QA checks
# -------------------------------------------------------------
assert gdf.duplicated(subset=["name", "address"]).sum() == 0
assert gdf.geometry.is_valid.all()
print(f"[INFO] Total dental office records after deduplication: {gdf.shape[0]}")

# -------------------------------------------------------------
# Step 7: Duplicate report by Name + Address (diagnostic)
# -------------------------------------------------------------
df_final_dupl = (
    gdf.groupby(["name", "address"], dropna=False)
       .size()
       .reset_index(name="count_duplicates")
       .sort_values("count_duplicates", ascending=False)
)


# -------------------------------------------------------------
# Step 8: Final centroid-based deduplication (~10 m) + Name
# -------------------------------------------------------------
MAX_DISTANCE_DEGREES = 0.0001  # ~10 m
sindex = gdf.sindex
duplicates = set()
clusters = []

# Normalized name column for comparison
if "name_norm" not in gdf.columns:
    gdf["name_norm"] = gdf["name"].apply(lambda x: str(x).lower().strip())

for idx, row in gdf.iterrows():
    if idx in duplicates:
        continue
    # Query candidates in buffer (~10 m)
    candidates = list(sindex.query(row.geometry.buffer(MAX_DISTANCE_DEGREES)))
    cluster = [idx]
    for cand_idx in candidates:
        if cand_idx <= idx or cand_idx in duplicates:
            continue
        other = gdf.iloc[cand_idx]
        if (
            row.geometry.distance(other.geometry) <= MAX_DISTANCE_DEGREES
            and row["name_norm"] == other["name_norm"]
        ):
            cluster.append(cand_idx)
            duplicates.add(cand_idx)
    clusters.append(cluster)

# -------------------------------------------------------------
# Step 9: Generate duplicate report for centroid-based duplicates
# -------------------------------------------------------------
cluster_rows = []
for i, cluster in enumerate(clusters):
    if len(cluster) > 1:
        for idx in cluster:
            cluster_rows.append({
                "cluster_id": i,
                "original_index": idx,
                "name": gdf.loc[idx, "name"],
                "address": gdf.loc[idx, "address"],
                "latitude": gdf.loc[idx, "latitude"],
                "longitude": gdf.loc[idx, "longitude"]
            })

if cluster_rows:
    df_duplicates_report = pd.DataFrame(cluster_rows)
    df_duplicates_report = df_duplicates_report.sort_values(
        ["cluster_id", "original_index"]
    ).reset_index(drop=True)
else:
    df_duplicates_report = pd.DataFrame(
        columns=["cluster_id", "original_index", "name", "address", "latitude", "longitude"]
    )

print("[INFO] Centroid-based duplicate report:")
display(df_duplicates_report.head())


# -------------------------------------------------------------
# Step 10: Merge clusters to final deduplicated GeoDataFrame
# -------------------------------------------------------------
final_rows = [merge_rows_safely(gdf.iloc[cluster]) for cluster in clusters]
gdf_final = gpd.GeoDataFrame(
    final_rows,
    geometry="geometry",
    crs=WGS84
).reset_index(drop=True)

# Final QA
assert gdf_final.geometry.is_valid.all()
print(f"[INFO] Total records after final centroid-based deduplication: {len(gdf_final)}")

# -------------------------------------------------------------
# Step 11: Filter original df_final to only keep deduplicated IDs
# -------------------------------------------------------------
df_final = df_final[df_final["id"].isin(gdf_final["id"])].copy()
df_final.reset_index(drop=True, inplace=True)


[INFO] Total dental office records after deduplication: 796
[INFO] Centroid-based duplicate report:


,cluster_id,original_index,name,address,latitude,longitude


[INFO] Total records after final centroid-based deduplication: 796


## ============================================================
## 2.2. Exclude Entries Already Present in Doctors Dataset
## ============================================================

In [153]:
df_doctors = pd.read_csv("../sources/doctors_202601211553.csv")
# -----------------------------
# Step 1: Find overlapping IDs
# -----------------------------
overlapping_ids = set(df_final['id']).intersection(set(df_doctors['id']))

print(f"[INFO] Number of overlapping IDs: {len(overlapping_ids)}")

# -----------------------------
# Step 2: If overlaps exist, remove them from df_final
# -----------------------------
if overlapping_ids:
    # Keep only rows whose dental office id is NOT in df_doctors
    df_final = df_final[~df_final['id'].isin(overlapping_ids)]
    print(f"[INFO] Removed {len(overlapping_ids)} overlapping dental office entries from df_final")
else:
    print("[INFO] No overlapping IDs found. df_final remains unchanged.")

[INFO] Number of overlapping IDs: 0
[INFO] No overlapping IDs found. df_final remains unchanged.


In [154]:
# -----------------------------
# Step 1: Find overlapping IDs
# -----------------------------
overlapping_ids = set(df_final['id']).intersection(set(df_doctors['id']))

print(f"[INFO] Number of overlapping IDs: {len(overlapping_ids)}")

# -----------------------------
# Step 2: If overlaps exist, remove them from df_final
# -----------------------------
if overlapping_ids:
    # Keep only rows whose dental office id is NOT in df_doctors
    df_final = df_final[~df_final['id'].isin(overlapping_ids)]
    print(f"[INFO] Removed {len(overlapping_ids)} overlapping dental office entries from df_final")
else:
    print("[INFO] No overlapping IDs found. df_final remains unchanged.")

[INFO] Number of overlapping IDs: 0
[INFO] No overlapping IDs found. df_final remains unchanged.


## ============================================================
## 2.3. Data Cleaning & Standardization Pipeline
## ============================================================

In [155]:
# ============================================================
# Column Name Standardization
# ============================================================
def to_snake_case(col: str) -> str:
    """
    Convert column names to snake_case:
    - Lowercase
    - Trim whitespace
    - Remove special characters
    - Replace spaces with underscores
    """
    col = col.lower().strip()
    col = re.sub(r"[^\w\s]", "", col)   # Remove special characters
    col = re.sub(r"\s+", "_", col)      # Replace whitespace with underscore
    return col


# Apply column name normalization
df_final.columns = [to_snake_case(c) for c in df_final.columns]

# Inspect dataframe structure after renaming
df_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 796 entries, 0 to 795
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               796 non-null    string  
 1   name             766 non-null    object  
 2   street           796 non-null    object  
 3   housenumber      609 non-null    object  
 4   postcode         796 non-null    object  
 5   city             796 non-null    object  
 6   level            91 non-null     object  
 7   description      32 non-null     object  
 8   opening_hours    601 non-null    object  
 9   check_date       284 non-null    object  
 10  accessibility    286 non-null    object  
 11  phone            459 non-null    object  
 12  email            137 non-null    object  
 13  website          425 non-null    object  
 14  geometry         796 non-null    geometry
 15  office_type      23 non-null     object  
 16  speciality       112 non-null    obj

In [156]:
df_final['speciality'].unique()

array([None, 'dentist', 'orthodontics', 'implantology',
       'dental_oral_maxillo_facial_surgery', 'stomatology', 'paediatrics',
       'endodontics;parodontology;acupuncture', 'dentist;orthodontics',
       'endodontics;Individualprophylaxe;Vollkeramikrestaurationen;Ästhetik;implantatversorgung;modernste_Zahnerhaltung_(spez._Endodontie)',
       'paediatric_dentistry;endodontics', 'general', 'parodontology',
       'oral_surgery', 'dentistry', 'orthodontics;oral_surgery',
       'endodontics;implantology;paediatric_dentistry',
       'implantology;endodontics',
       'paediatric_dentistry;endodontics;bleaching',
       'oral_surgery;implantology', 'general;endodontics;implantology',
       'dental_oral_maxillo_facial_surgery;stomatology', 'endodontics',
       'dental_oral_maxillo_facial_surgery;orthodontics',
       'dentist;stomatology'], dtype=object)

In [157]:
# ============================================================
# Speciality Normalization
# ============================================================
def normalize_speciality(val):
    """
    Normalize speciality values:
    - Handle missing values
    - Convert to lowercase
    - Normalize separators
    - Split multiple entries
    - Deduplicate and sort
    - Return a pipe ('|') separated string
    """
    if pd.isna(val):
        return ""

    # Normalize casing and whitespace
    val = str(val).lower().strip()

    # Replace common separators with a single delimiter
    val = re.sub(r"[,\|/]", ";", val)

    # Split, clean, deduplicate, and sort
    items = [x.strip() for x in val.split(";") if x.strip()]
    items_unique = sorted(set(items))

    # Join into a single string
    return "|".join(items_unique)


# Apply speciality normalization
df_final["speciality"] = df_final["speciality"].apply(normalize_speciality)

# Validate results
print(df_final["speciality"].value_counts())



speciality
                                                                                                                                      684
orthodontics                                                                                                                           51
dentist                                                                                                                                16
stomatology                                                                                                                             7
general                                                                                                                                 6
implantology                                                                                                                            4
oral_surgery                                                                                                                            4
dentistry              

In [158]:
print(f"Unique values in 'accessibility':\n {df_final['accessibility'].unique()}\n")
print(f"Unique values in 'description':\n {df_final['description'].unique()}\n")
print(f"Unique values in 'check_date':\n {df_final['check_date'].unique()}\n")


Unique values in 'accessibility':
 [None 'wheelchair: yes' 'wheelchair: yes | wheelchair toilets: no'
 'wheelchair: no' 'wheelchair: limited'
 'wheelchair: no | wheelchair toilets: no'
 'wheelchair: yes | wheelchair toilets: yes'
 'wheelchair: limited | wheelchair toilets: no']

Unique values in 'description':
 [None 'Zahnarzt' 'Zahnärzte & Kieferorthopädie, Aufgang G'
 'Fahrstuhl und Türöffner vorhanden'
 'Zahnarztpraxis Ursula Frömming, Ruth Bodenheimer'
 'Praxis für ästhetisch-rekonstruktive Zahnmedizin und Implantologie'
 'Kieferorthopädin'
 'Sehr nette Fachärztin :) Öffnungszeiten: Mo: 8:00 - 17:00 Di: 8:00 - 19:00 Mi: 8:00 - 19:00 Do: 8:00 - 19:00 Fr: 8:00 - 16:00'
 'Aufzugtür nur 70cm breit'
 'Öffnungszeiten: Mo - Fr 9:00 - 19:00 Uhr Fr nur mit Termin'
 'Zahnarztpraxis Reinickendorf, Zahnarzt Reinickendorf, Implantation Reinickendorf, Zahnarzt Wittenau'
 'Barrierefreier Zugang durch feste Rampe und Lift. Die Praxis verfügt über ein behindertengerechtes WC.'
 'Dipl.-Stomatologin'

In [159]:
# ============================================================
# Generic String Normalization
# ============================================================
def normalize_strings(val):
    """
    Normalize generic string columns:
    - Handle missing values
    - Convert to lowercase
    - Strip leading and trailing whitespace
    """
    if pd.isna(val):
        return None

    return str(val).lower().strip()


# Normalize accessibility column
df_final["accessibility"] = df_final["accessibility"].apply(normalize_strings)

# Normalize description column
df_final["description"] = df_final["description"].apply(normalize_strings)

# Normalize check_date column (kept as string by design)
df_final["check_date"] = df_final["check_date"].apply(normalize_strings)

df_final['name'] = df_final['name'].fillna('unknown dentist')
# Normalize phone numbers
df_final["phone"] = (
    df_final["phone"]
    .str.replace(r"[^\d+]", "", regex=True)
)

In [160]:
df_final.columns.tolist()

['id',
 'name',
 'street',
 'housenumber',
 'postcode',
 'city',
 'level',
 'description',
 'opening_hours',
 'check_date',
 'accessibility',
 'phone',
 'email',
 'website',
 'geometry',
 'office_type',
 'speciality',
 'latitude',
 'longitude',
 'district',
 'neighborhood',
 'neighborhood_id',
 'district_id',
 'address']

In [161]:
dental_offices = df_final[
    [
        "id",
        "name",
        "address",
        "phone",
        "email",
        "website",
        "opening_hours",
        "check_date",
        "accessibility",
        "speciality",
        "description",
        "office_type",
        "geometry",
        "latitude",
        "longitude",
        "district",
        "neighborhood",
        "district_id",
        "neighborhood_id",
    ]
]

## ============================================================
## 2.4. Missing Value Validation & Default Imputation
## ============================================================

In [162]:
# ============================================================
# Define default values for missing data
# ============================================================
DEFAULT_VALUES = {
    "name": "Unknown",
    "address": None,
    "phone": None,
    "email": None,
    "website": None,
    "opening_hours": None,
    "check_date": None,
    "accessibility": None,
    "speciality": None,
    "description": None,
    "office_type": None,
}

# Columns that must be validated
COLUMNS_TO_VALIDATE = [
    "name",
    "address",
    "phone",
    "email",
    "website",
    "opening_hours",
    "check_date",
    "accessibility",
    "speciality",
    "description",
    "office_type",
    "geometry",
    "latitude",
    "longitude",
    "district",
    "neighborhood",
    "district_id",
    "neighborhood_id",
]

# ============================================================
# Validate column existence
# ============================================================
missing_columns = set(COLUMNS_TO_VALIDATE) - set(df_final.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# ============================================================
# Impute missing values where defaults are defined
# ============================================================
for column, default_value in DEFAULT_VALUES.items():

    if default_value is None:
        # Explicitly convert NaN to None (SQL NULL semantics)
        df_final.loc[df_final[column].isna(), column] = None
    else:
        # Replace NaN with actual default values
        df_final.loc[:, column] = df_final[column].fillna(default_value)


# ============================================================
# Report remaining NaN values (non-imputed columns)
# ============================================================
nan_report = dental_offices[COLUMNS_TO_VALIDATE].isna().sum()
nan_report = nan_report[nan_report > 0]

if not nan_report.empty:
    print("Remaining NaN values detected:")
    print(nan_report)
else:
    print("No remaining NaN values in validated columns.")
print('Data types:')
display(dental_offices.dtypes)
print(dental_offices['name'].nunique(), "unique dental office names")

Remaining NaN values detected:
phone            337
email            659
website          371
opening_hours    195
check_date       512
accessibility    510
description      764
office_type      773
dtype: int64
Data types:


id                 string[python]
name                       object
address                    object
phone                      object
email                      object
website                    object
opening_hours              object
check_date                 object
accessibility              object
speciality                 object
description                object
office_type                object
geometry                 geometry
latitude                  float64
longitude                 float64
district                   object
neighborhood               object
district_id        string[python]
neighborhood_id            object
dtype: object

745 unique dental office names


## ============================================================
## 2.5 Final quality checks for the GeoDataFrame
## ============================================================

In [163]:
# Check that all geometries are valid (no corrupted or empty geometries)
assert dental_offices.geometry.is_valid.all(), "Some geometries in the GeoDataFrame are invalid."

# Verify that the coordinate reference system (CRS) is WGS84
assert dental_offices.crs.to_string() == WGS84, f"CRS mismatch: expected {WGS84}, got {dental_offices.crs.to_string()}"

# Ensure that all dental offices have a non-missing name
assert dental_offices["name"].notna().all(), "There are missing values in the 'name' column."

# -------------------------------------------------------------
# Validate that all points fall within Berlin’s geographic boundaries
# -------------------------------------------------------------

# Define approximate bounding box for Berlin (WGS84)
BERLIN_BOUNDS = {
    "lat_min": 52.3,
    "lat_max": 52.7,
    "lon_min": 13.0,
    "lon_max": 13.8
}
# Assert latitude is within bounds
assert dental_offices["latitude"].between(BERLIN_BOUNDS["lat_min"], BERLIN_BOUNDS["lat_max"]).all(), \
    "Some latitude values fall outside Berlin's boundaries."

# Assert longitude is within bounds
assert dental_offices["longitude"].between(BERLIN_BOUNDS["lon_min"], BERLIN_BOUNDS["lon_max"]).all(), \
    "Some longitude values fall outside Berlin's boundaries."



# =============================================================
## 2.6 Safely prepare GeoDataFrame geometry for database insertion
# =============================================================

In [164]:
# -------------------------------------------------------------
# Keep a copy of the original GeoDataFrame
# -------------------------------------------------------------
# We retain the original GeoDataFrame in case we need it later.
gdf_original = dental_offices.copy()  

# -------------------------------------------------------------
# Display the type of the object before conversion
# -------------------------------------------------------------
print(f"Original type: {type(dental_offices)}")

# -------------------------------------------------------------
# Convert the geometry column to WKT for PostgreSQL compatibility
# -------------------------------------------------------------
# Important: The object remains a GeoDataFrame.
# Only the geometry column's data type is changed from shapely geometries
# to WKT strings to facilitate database insertion and SQL operations.
if hasattr(gdf_original, 'geometry'):
    print("Converting geometry from original GeoDataFrame to WKT string...")

    # Replace geometry with WKT string representation
    dental_offices['geometry'] = gdf_original.geometry.to_wkt()

    print(f"Geometry after conversion: {type(dental_offices['geometry'].iloc[0])}")
    print(f"Example geometry: {dental_offices['geometry'].iloc[0][:50]}...")

# -------------------------------------------------------------
# Confirm the GeoDataFrame is still intact
# -------------------------------------------------------------
print(f"Final type: {type(dental_offices)}")

Original type: <class 'geopandas.geodataframe.GeoDataFrame'>
Converting geometry from original GeoDataFrame to WKT string...
Geometry after conversion: <class 'str'>
Example geometry: POINT (13.612096 52.511411)...
Final type: <class 'geopandas.geodataframe.GeoDataFrame'>


In [165]:
dental_offices.head()

,id,name,address,phone,email,website,opening_hours,check_date,accessibility,speciality,description,office_type,geometry,latitude,longitude,district,neighborhood,district_id,neighborhood_id
0,304183504,unknown dentist,"Hönower Straße 75, 12623 Berlin-Mahlsdorf",None,None,None,NaN,None,None,,None,office,POINT (13.612096 52.511411),52.511411,13.612096,Marzahn-Hellersdorf,Mahlsdorf,11010010,1004
1,313539258,Zahnzentrum Wedding,"Müllerstraße 34a, 2.OG, 13353 Berlin-Wedding",None,None,None,Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...,None,wheelchair: yes,,zahnarzt,NaN,POINT (13.355305 52.548838),52.548838,13.355305,Mitte,Wedding,11001001,0105
2,325161442,A. Nejad,"Gatower Straße, 13595 Berlin-Wilhelmstadt",+49303619106,None,None,Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...,None,wheelchair: yes | wheelchair toilets: no,,None,NaN,POINT (13.180477 52.508843),52.508843,13.180477,Spandau,Wilhelmstadt,11005005,0509
3,345236220,Dr. Beate Lengert,"Kurfürstendamm 218, 10719 Berlin-Charlottenburg",None,None,http://www.dr-beate-lengert.de/,NaN,None,None,,None,NaN,POINT (13.328137 52.502722),52.502722,13.328137,Charlottenburg-Wilmersdorf,Charlottenburg,11004004,0401
4,391394177,Serpil Hartfiel,"Kollwitzstraße 77, 10435 Berlin-Prenzlauer Berg",None,None,https://www.zahnarztpraxis-hartfiel.de/,"Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...",None,wheelchair: no,,None,NaN,POINT (13.418994 52.537547),52.537547,13.418994,Pankow,Prenzlauer Berg,11003003,0301


# ==============================
# Step III: Production Load (Dental Offices Data Layer)
# ==============================
## 3.1 Database Setup & Connection
# ==============================

In [166]:
# This code loads environment variables (from dental_offices/.env), establishes a secure database connection, and prepares the SQLAlchemy engine.
load_dotenv()  

USER_NAME = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASS")
HOST = os.getenv("DB_HOST", "localhost")
PORT = os.getenv("DB_PORT", "5433")
DATABASE = os.getenv("DB_NAME")
DB_SCHEMA = os.getenv("DB_SCHEMA")
TEST_TABLE = os.getenv("DB_TEST_TABLE")
DISTRICT_TABLE = os.getenv("DB_DISTRICT_TABLE")

# Create SQLAlchemy engine (PostgreSQL via psycopg2)
engine = create_engine(f'postgresql+psycopg2://{USER_NAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}', connect_args={"sslmode": "require"})
print("✅ Database engine created")

✅ Database engine created


# =============================================================
## 3.2 Temporary Test Table (Validation & Debugging) 
# =============================================================

### 3.2.1 CREATE TEST TABLE
# =============================================================



In [ ]:
#This table is used to validate schema design, data integrity,and foreign key behavior before populating the final table.
# -------------------------------------------------------------
drop_test_sql = f"DROP TABLE IF EXISTS {TEST_TABLE};"

create_test_sql = f"""
CREATE TABLE {TEST_TABLE} (
    id VARCHAR(20) PRIMARY KEY,
    name VARCHAR(200) NOT NULL DEFAULT 'Unknown',
    address VARCHAR(255),
    phone VARCHAR(50),
    email VARCHAR(100),
    website VARCHAR(200),
    opening_hours VARCHAR(200),
    check_date VARCHAR(255),
    accessibility VARCHAR(100),
    speciality VARCHAR(150),
    description VARCHAR(500),
    office_type VARCHAR(100),
    geometry VARCHAR,
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    district VARCHAR(100),
    neighborhood VARCHAR(100),
    district_id VARCHAR(20) NOT NULL,
    neighborhood_id VARCHAR(20),
 
    -- constraints
    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES berlin_source_data.districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE,

    CONSTRAINT latitude_check
        CHECK (latitude BETWEEN 52.3 AND 52.7),

    CONSTRAINT longitude_check
        CHECK (longitude BETWEEN 13.0 AND 13.9)
)
"""
# Execute table creation
with engine.begin() as conn:
    conn.execute(text(drop_test_sql))
    conn.execute(text(create_test_sql))

print("✅ Test table created")


# =============================================================
### 3.2.2 DATA INSERTION (TEST TABLE)
# =============================================================

In [ ]:
# Inserts the prepared DataFrame into the test table using pandas.to_sql for fast bulk loading.
dental_offices.to_sql(
    name=TEST_TABLE,
    con=engine,
    if_exists="append",
    index=False
)

print(f"✅ Inserted {len(dental_offices)} rows into {TEST_TABLE}")

✅ Inserted 796 rows into test_table_alexandra_dernova


# =============================================================
### 3.2.3. FOREIGN KEY VALIDATION & DATA INTEGRITY CHECKS (TEST TABLE)
# =============================================================
**This section verifies:**
1. FK existence
2. Absence of orphan records
3. FK enforcement at insert time
4. Final row count
# =============================================================


In [ ]:
# -------------------------------------------------------------
# 1 Verify Foreign Key Constraint Exists (Test Table)
# -------------------------------------------------------------
fk_exists_sql = f"""
SELECT
    conname AS constraint_name,
    a.attname AS column_name,
    confrelid::regclass AS referenced_table
FROM pg_constraint c
JOIN pg_attribute a
  ON a.attnum = ANY (c.conkey)
 AND a.attrelid = c.conrelid
WHERE c.contype = 'f'
  AND conrelid = '{TEST_TABLE}'::regclass;
"""

with engine.connect() as conn:
    fk_rows = conn.execute(text(fk_exists_sql)).fetchall()

if not fk_rows:
    raise RuntimeError("❌ Foreign key constraint NOT found")

print("✅ Foreign key constraint exists")
for row in fk_rows:
    print(dict(row._mapping))

# -------------------------------------------------------------
# 2 Check for Orphan Records
# -------------------------------------------------------------
orphan_check_sql = f"""
SELECT COUNT(*) 
FROM {TEST_TABLE} t
LEFT JOIN {DISTRICT_TABLE} d
       ON t.district_id = d.district_id
WHERE d.district_id IS NULL;
"""

with engine.connect() as conn:
    orphan_count = conn.execute(text(orphan_check_sql)).scalar()

if orphan_count != 0:
    raise RuntimeError(f"❌ Found {orphan_count} orphan district_id records")

print("✅ No orphan district_id records found")

# -------------------------------------------------------------
# 3 Validate FK Enforcement
# -------------------------------------------------------------

invalid_insert_sql = f"""
INSERT INTO {TEST_TABLE} (id, district_id, name, district)
VALUES ('fk_test_999', '99999999', 'FK Test', 'Invalid');
"""

try:
    with engine.begin() as conn:
        conn.execute(text(invalid_insert_sql))
    raise RuntimeError("❌ FK enforcement FAILED — invalid insert allowed")
except IntegrityError:
    print("✅ Foreign key enforcement confirmed")

# -------------------------------------------------------------
# 4 Final Row Count Check
# -------------------------------------------------------------
with engine.connect() as conn:
    row_count = conn.execute(text(f"SELECT COUNT(*) FROM {TEST_TABLE};")).scalar()

print(f"📊 Total rows in test table: {row_count}")
print("\n🎉 ALL TEST TABLE CHECKS PASSED SUCCESSFULLY")

✅ Foreign key constraint exists
{'constraint_name': 'district_id_fk', 'column_name': 'district_id', 'referenced_table': 'berlin_source_data.districts'}
✅ No orphan district_id records found
✅ Foreign key enforcement confirmed
📊 Total rows in test table: 796

🎉 ALL TEST TABLE CHECKS PASSED SUCCESSFULLY


# =============================================================
## 3.3 Final Production Table: dental_offices
# =============================================================
**This is the final, normalized production table populated after all validation steps succeeded.**

### 3.3.1 CREATE FINAL TABLE
# =============================================================

In [ ]:
FINAL_TABLE = f"{DB_SCHEMA}.dental_offices"

drop_final_sql = f"DROP TABLE IF EXISTS {FINAL_TABLE};"

create_final_sql = f"""
CREATE TABLE {FINAL_TABLE} (
    id VARCHAR(20) PRIMARY KEY,
    name VARCHAR(200) NOT NULL DEFAULT 'Unknown',
    address VARCHAR(255),
    phone VARCHAR(50),
    email VARCHAR(100),
    website VARCHAR(200),
    opening_hours VARCHAR(200),
    check_date VARCHAR(255),
    accessibility VARCHAR(100),
    speciality VARCHAR(150),
    description VARCHAR(500),
    office_type VARCHAR(100),
    geometry VARCHAR,
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    district VARCHAR(100),
    neighborhood VARCHAR(100),
    district_id VARCHAR(20) NOT NULL,
    neighborhood_id VARCHAR(20),
    
    -- Constraints
    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES berlin_source_data.districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE,

    CONSTRAINT latitude_check
        CHECK (latitude BETWEEN 52.3 AND 52.7),
    CONSTRAINT longitude_check
        CHECK (longitude BETWEEN 13.0 AND 13.9)
);
"""

with engine.begin() as conn:
    conn.execute(text(drop_final_sql))
    conn.execute(text(create_final_sql))

print("✅ Final table dental_offices created")

✅ Final table dental_offices created


# =============================================================
### 3.3.2 DATA INSERTION (FINAL TABLE)
# =============================================================

In [ ]:
dental_offices.to_sql(
    name="dental_offices",
    con=engine,
    schema=DB_SCHEMA,
    if_exists="append",
    index=False
)

print(f"✅ Inserted {len(dental_offices)} rows into dental_offices")
print("\n🚀 DATABASE POPULATION COMPLETED SUCCESSFULLY")

✅ Inserted 796 rows into dental_offices

🚀 DATABASE POPULATION COMPLETED SUCCESSFULLY


# =============================================================
## 3.4 Post-Load Data Quality Checks (Final Table)
# =============================================================
**This section performs analytical sanity checks directly on the production table to validate data consistency.**
# =============================================================

### 3.4.1 DATABASE READABILITY CHECK
# =============================================================

In [ ]:
print("\n🔍 RUNNING POST-LOAD DATA QUALITY CHECKS\n")
# -------------------------------------------------------------
#  Unique Addresses per District
# -------------------------------------------------------------
# Counts distinct addresses per district to detect
# duplicates, missing data, or anomalies.
# -------------------------------------------------------------
unique_addresses_sql = f"""
SELECT
    district,
    COUNT(DISTINCT address) AS unique_addresses
FROM {FINAL_TABLE}
WHERE address IS NOT NULL
GROUP BY district
ORDER BY unique_addresses DESC;
"""

with engine.connect() as conn:
    result = conn.execute(text(unique_addresses_sql))
    rows = result.fetchall()

# Pretty console output
print("📊 Unique addresses per district:\n")
for district, count in rows:
    print(f"• {district:<25} {count:>5}")

print("\n✅ Address uniqueness check completed")


🔍 RUNNING POST-LOAD DATA QUALITY CHECKS

📊 Unique addresses per district:

• Charlottenburg-Wilmersdorf   130
• Tempelhof-Schöneberg        106
• Pankow                       90
• Mitte                        81
• Steglitz-Zehlendorf          66
• Neukölln                     66
• Friedrichshain-Kreuzberg     57
• Reinickendorf                51
• Treptow-Köpenick             38
• Lichtenberg                  36
• Marzahn-Hellersdorf          23
• Spandau                      17

✅ Address uniqueness check completed


# =============================================================
### 3.4.2 FOREIGN KEY VALIDATION (FINAL TABLE)
# =============================================================

In [ ]:
# -------------------------------------------------------------
# 1 Verify Foreign Key Constraint Exists (Test Table)
# -------------------------------------------------------------

fk_exists_sql = f"""
SELECT
    conname AS constraint_name,
    a.attname AS column_name,
    confrelid::regclass AS referenced_table
FROM pg_constraint c
JOIN pg_attribute a
  ON a.attnum = ANY (c.conkey)
 AND a.attrelid = c.conrelid
WHERE c.contype = 'f'
  AND conrelid = '{FINAL_TABLE}'::regclass;
"""

with engine.connect() as conn:
    fk_rows = conn.execute(text(fk_exists_sql)).fetchall()

if not fk_rows:
    raise RuntimeError("❌ Foreign key constraint NOT found")

print("✅ Foreign key constraint exists")
for row in fk_rows:
    print(dict(row._mapping))

# -------------------------------------------------------------
# 2 Check for Orphan Records
# -------------------------------------------------------------
orphan_check_sql = f"""
SELECT COUNT(*) 
FROM {FINAL_TABLE} t
LEFT JOIN {DISTRICT_TABLE} d
       ON t.district_id = d.district_id
WHERE d.district_id IS NULL;
"""

with engine.connect() as conn:
    orphan_count = conn.execute(text(orphan_check_sql)).scalar()

if orphan_count != 0:
    raise RuntimeError(f"❌ Found {orphan_count} orphan district_id records")

print("✅ No orphan district_id records found")

# -------------------------------------------------------------
# 3 Validate FK Enforcement
# -------------------------------------------------------------
invalid_insert_sql = f"""
INSERT INTO {FINAL_TABLE} (id, district_id, name, district)
VALUES ('fk_test_999', '99999999', 'FK Test', 'Invalid');
"""

try:
    with engine.begin() as conn:
        conn.execute(text(invalid_insert_sql))
    raise RuntimeError("❌ FK enforcement FAILED — invalid insert allowed")
except IntegrityError:
    print("✅ Foreign key enforcement confirmed")




✅ Foreign key constraint exists
{'constraint_name': 'district_id_fk', 'column_name': 'district_id', 'referenced_table': 'berlin_source_data.districts'}
✅ No orphan district_id records found
✅ Foreign key enforcement confirmed


# =============================================================
### 3.4.3 Final Row Count Check
# =============================================================

In [ ]:
with engine.connect() as conn:
    row_count = conn.execute(text(f"SELECT COUNT(*) FROM {FINAL_TABLE};")).scalar()

print(f"📊 Total rows in final table: {row_count}")
print("\n🎉 ALL FINAL TABLE CHECKS PASSED SUCCESSFULLY")

📊 Total rows in final table: 796

🎉 ALL FINAL TABLE CHECKS PASSED SUCCESSFULLY
